# §13.5.4 — 마스크 누락이 손실을 낮추는 것의 재현

> 딥러닝 교재 · 3부 13장 5절 4항 (🐍)
> 선행: §13.5.1(마스크 정의) · §13.5.2(병렬 학습) · §13.5.5(위생 검사)

## 이 노트북이 답하는 질문

1. **인과 마스크를 빠뜨리면 학습 손실은 어떻게 되는가?** 나빠지는 게 아니라 좋아진다.
2. **그 "좋은" 모델은 생성에서 어떻게 되는가?** 미래가 사라지는 순간을 잰다.
3. **버그를 어떻게 잡는가?** 레이블 셔플 위생 검사를 실행한다.

**예상 실행 시간** CPU 약 3분 (`FAST = True`이면 약 90초).

---
## 0. 설정

In [ ]:
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

_t0 = time.time()

# ── 손잡이 (마지막 셀에 전체 목록) ──────────────────────────
FAST     = False     # True면 시행 수를 줄여 빠르게
SEED     = 20260808
SAVE_PDF = False     # True면 figs/에 교재용 PDF 저장
FIG_DIR  = 'figs'
# ──────────────────────────────────────────────────────────

# 색맹 안전 팔레트 (Okabe–Ito) — 규약 §II.4-7
CB = ['#000000', '#E69F00', '#56B4E9', '#009E73',
      '#D55E00', '#0072B2', '#CC79A7', '#F0E442']
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'axes.grid': True,
                     'grid.alpha': 0.3, 'axes.prop_cycle': plt.cycler(color=CB),
                     'figure.autolayout': True})

# 한글 폰트 (부록 K). 없으면 그림 라벨만 영문으로 대체한다.
for _p in glob.glob('/usr/share/fonts/**/*CJK*.ttc', recursive=True)[:6]:
    try:
        fm.fontManager.addfont(_p)
    except Exception:
        pass
_av = {f.name for f in fm.fontManager.ttflist}
KO_FONT = next((f for f in ['NanumGothic', 'Malgun Gothic', 'AppleGothic',
                            'Noto Sans CJK KR', 'Noto Sans KR', 'NanumBarunGothic',
                            'Noto Sans CJK JP'] if f in _av), None)
if KO_FONT:
    plt.rcParams['font.family'] = KO_FONT
    plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
def lab(ko, en):
    return ko if KO_FONT else en

def save_book_fig(fig, name):
    # 교재 결합 그림 저장 — 불필요한 여백 없이
    if SAVE_PDF:
        os.makedirs(FIG_DIR, exist_ok=True)
        fig.savefig(os.path.join(FIG_DIR, name + '.pdf'),
                    bbox_inches='tight', pad_inches=0.03)
        print('저장:', os.path.join(FIG_DIR, name + '.pdf'))

rng = np.random.default_rng(SEED)
print(f"numpy {np.__version__}  |  FAST={FAST}  |  seed={SEED}")
print(f"한글 폰트: {KO_FONT or '없음 → 그림 라벨은 영문으로 출력됩니다'}")

---
## 1. 데이터와 네 개의 학습 실행

데이터는 상태 20개짜리 마르코프 연쇄다. 상태마다 후속 4개에 [0.5, 0.25, 0.15, 0.1]의
확률을 주므로, 전이의 조건부 엔트로피 $\approx 1.28$ 나트가 **정직한 모델이 도달할 수
있는 손실의 하한**이다. 이 하한보다 뚜렷이 낮은 학습 손실은 그 자체로 누출의 증거가
된다 — 미래를 보지 않고는 불가능한 숫자이기 때문이다.

실행은 넷: (마스크 O/X) × (정상 레이블 / 셔플 레이블). 뒤의 둘이 §13.5.5의 위생 검사다.

In [ ]:
# ── 공용 미니 트랜스포머 (NumPy, 완전한 순전파+역전파) ──────────
# 구조: 임베딩 → L × [Pre-LN 블록 (MHA + MLP, 잔차)] → LN → 판독
# 위치 부호화: 'learned' | 'sin' | 'rope' | 'alibi' | 'none'

def make_config(V, d=48, L=2, h=4, T_max=64, pe='learned', causal=True, seed=0):
    dh = d // h
    rn = np.random.default_rng(seed)
    p = {}
    p['emb'] = rn.standard_normal((V, d)) * 0.5 / np.sqrt(d)
    if pe == 'learned':
        p['pos'] = rn.standard_normal((T_max, d)) * 0.5 / np.sqrt(d)
    for l in range(L):
        s = f'l{l}_'
        for nm in ['wq', 'wk', 'wv', 'wo']:
            p[s + nm] = rn.standard_normal((d, d)) / np.sqrt(d)
        p[s + 'ln1g'] = np.ones(d); p[s + 'ln1b'] = np.zeros(d)
        p[s + 'w1'] = rn.standard_normal((d, 4 * d)) / np.sqrt(d)
        p[s + 'b1'] = np.zeros(4 * d)
        p[s + 'w2'] = rn.standard_normal((4 * d, d)) / np.sqrt(4 * d)
        p[s + 'b2'] = np.zeros(d)
        p[s + 'ln2g'] = np.ones(d); p[s + 'ln2b'] = np.zeros(d)
    p['lnfg'] = np.ones(d); p['lnfb'] = np.zeros(d)
    p['out'] = rn.standard_normal((d, V)) / np.sqrt(d)
    cfg = dict(V=V, d=d, L=L, h=h, dh=dh, T_max=T_max, pe=pe, causal=causal)
    return p, cfg

def _sin_pe(T, d):
    pos = np.arange(T)[:, None]
    l2 = np.arange(0, d, 2)[None, :]
    ang = pos / (10000.0 ** (l2 / d))
    pe = np.zeros((T, d))
    pe[:, 0::2] = np.sin(ang); pe[:, 1::2] = np.cos(ang)
    return pe

def _rope_angles(T, dh, scale=1.0):
    pos = np.arange(T)[:, None] * scale
    l2 = np.arange(0, dh, 2)[None, :]
    return pos / (10000.0 ** (l2 / dh))          # (T, dh/2)

def _rope_apply(x, ang, inverse=False):
    # x: (B,h,T,dh) — 짝수/홀수 쌍을 각도 ang(T,dh/2)만큼 회전
    c, s = np.cos(ang), np.sin(ang)
    if inverse:
        s = -s
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return np.stack([x1 * c - x2 * s, x1 * s + x2 * c], axis=-1).reshape(x.shape)

def _alibi_slopes(h):
    return np.array([2.0 ** (-8.0 * (i + 1) / h) for i in range(h)])

def _ln_f(x, g, b):
    mu = x.mean(-1, keepdims=True)
    xc = x - mu
    var = (xc ** 2).mean(-1, keepdims=True)
    inv = 1.0 / np.sqrt(var + 1e-5)
    xh = xc * inv
    return xh * g + b, (xh, inv)

def _ln_b(dy, cache, g):
    xh, inv = cache
    dxh = dy * g
    dg = (dy * xh).sum(axis=tuple(range(dy.ndim - 1)))
    db = dy.sum(axis=tuple(range(dy.ndim - 1)))
    dx = inv * (dxh - dxh.mean(-1, keepdims=True) - xh * (dxh * xh).mean(-1, keepdims=True))
    return dx, dg, db

def forward(p, cfg, idx, targets=None, rope_scale=1.0, head_mask=None,
            skip=None, want_attn=False, kv_keep=None):
    """idx:(B,T) 정수. targets:(B,T) 또는 None.
    head_mask:(L,h) 0/1, skip: {'attn':set(l), 'mlp':set(l)},
    kv_keep:(T,) bool — 열 s의 키·값 사용 여부(캐시 축출 흉내)."""
    B, T = idx.shape
    d, L, h, dh = cfg['d'], cfg['L'], cfg['h'], cfg['dh']
    x = p['emb'][idx]                              # (B,T,d)
    if cfg['pe'] == 'learned':
        x = x + p['pos'][:T]
    elif cfg['pe'] == 'sin':
        x = x + _sin_pe(T, d)
    cache = {'idx': idx, 'T': T, 'B': B, 'xs': [], 'attn': []}
    ang = _rope_angles(T, dh, rope_scale) if cfg['pe'] == 'rope' else None
    if cfg['causal']:
        nmask = np.triu(np.full((T, T), -np.inf), k=1)
    else:
        nmask = np.zeros((T, T))
    if kv_keep is not None:
        nmask = nmask.copy()
        nmask[:, ~kv_keep] = -np.inf
    if cfg['pe'] == 'alibi':
        sl = _alibi_slopes(h)
        dist = np.maximum(np.arange(T)[:, None] - np.arange(T)[None, :], 0)
        abias = -sl[:, None, None] * dist[None]    # (h,T,T)
    else:
        abias = np.zeros((1, T, T))
    skip = skip or {'attn': set(), 'mlp': set()}
    attns = []
    def split(z):
        return z.reshape(B, T, h, dh).transpose(0, 2, 1, 3)       # (B,h,T,dh)
    for l in range(L):
        s = f'l{l}_'
        c = {}
        if l not in skip['attn']:
            # ── MHA 가지 ──
            h1, c['ln1'] = _ln_f(x, p[s + 'ln1g'], p[s + 'ln1b'])
            c['h1'] = h1
            q = split(h1 @ p[s + 'wq']); k = split(h1 @ p[s + 'wk']); v = split(h1 @ p[s + 'wv'])
            if cfg['pe'] == 'rope':
                q = _rope_apply(q, ang); k = _rope_apply(k, ang)
            e = np.einsum('bhtd,bhsd->bhts', q, k) / np.sqrt(dh) + nmask + abias[None]
            e -= e.max(-1, keepdims=True)
            a = np.exp(e); a /= a.sum(-1, keepdims=True)
            if head_mask is not None:
                hm = head_mask[l][None, :, None, None]
            else:
                hm = 1.0
            av = np.einsum('bhts,bhsd->bhtd', a, v) * hm
            avm = av.transpose(0, 2, 1, 3).reshape(B, T, d)
            x = x + avm @ p[s + 'wo']
            c.update(q=q, k=k, v=v, a=a, avm=avm, hm=hm)
            attns.append(a)
        else:
            attns.append(None)
        if l not in skip['mlp']:
            # ── MLP 가지 ──
            h2, c['ln2'] = _ln_f(x, p[s + 'ln2g'], p[s + 'ln2b'])
            z1 = h2 @ p[s + 'w1'] + p[s + 'b1']
            r = np.maximum(z1, 0.0)               # ReLU (역전파 단순화)
            x = x + r @ p[s + 'w2'] + p[s + 'b2']
            c['mlp'] = (h2, z1, r)
        else:
            c['mlp'] = None
        cache[s] = c
    hf, cache['lnf'] = _ln_f(x, p['lnfg'], p['lnfb'])
    cache['hf'] = hf
    logits = hf @ p['out']
    cache['logits'] = logits
    out = {'logits': logits}
    if want_attn:
        out['attn'] = attns
    if targets is not None:
        valid = targets >= 0                      # -1 = 손실에서 제외
        tsafe = np.maximum(targets, 0)
        z = logits - logits.max(-1, keepdims=True)
        lse = np.log(np.exp(z).sum(-1))
        ll = z[np.arange(B)[:, None], np.arange(T)[None, :], tsafe] - lse
        out['loss'] = -(ll * valid).sum() / max(valid.sum(), 1)
        P = np.exp(z); P /= P.sum(-1, keepdims=True)
        cache['P'] = P; cache['targets'] = tsafe; cache['valid'] = valid
    out['cache'] = cache
    return out

def backward(p, cfg, cache, rope_scale=1.0):
    B, T = cache['B'], cache['T']
    d, L, h, dh = cfg['d'], cfg['L'], cfg['h'], cfg['dh']
    g = {k: np.zeros_like(v) for k, v in p.items()}
    P, targets, valid = cache['P'], cache['targets'], cache['valid']
    dlogits = P.copy()
    dlogits[np.arange(B)[:, None], np.arange(T)[None, :], targets] -= 1.0
    dlogits *= valid[:, :, None]
    dlogits /= max(valid.sum(), 1)
    hf = cache['hf']
    g['out'] = np.einsum('btd,btv->dv', hf, dlogits)
    dhf = dlogits @ p['out'].T
    dx, g['lnfg'], g['lnfb'] = _ln_b(dhf, cache['lnf'], p['lnfg'])
    ang = _rope_angles(T, dh, rope_scale) if cfg['pe'] == 'rope' else None
    for l in range(L - 1, -1, -1):
        s = f'l{l}_'
        c = cache[s]
        if c['mlp'] is not None:
            h2, z1, r = c['mlp']
            dmlp = dx                                   # 잔차: 가지로 흘러드는 기울기
            g[s + 'w2'] += np.einsum('btf,btd->fd', r, dmlp)
            g[s + 'b2'] += dmlp.sum((0, 1))
            dr = dmlp @ p[s + 'w2'].T
            dz1 = dr * (z1 > 0)
            g[s + 'w1'] += np.einsum('btd,btf->df', h2, dz1)
            g[s + 'b1'] += dz1.sum((0, 1))
            dh2 = dz1 @ p[s + 'w1'].T
            dxi, dg2, db2 = _ln_b(dh2, c['ln2'], p[s + 'ln2g'])
            g[s + 'ln2g'] += dg2; g[s + 'ln2b'] += db2
            dx = dx + dxi
        if 'a' not in c:
            continue
        # MHA 가지
        dattn_out = dx
        g[s + 'wo'] += np.einsum('btd,bte->de', c['avm'], dattn_out)
        davm = dattn_out @ p[s + 'wo'].T
        dav = davm.reshape(B, T, h, dh).transpose(0, 2, 1, 3) * c['hm']
        a, q, k, v = c['a'], c['q'], c['k'], c['v']
        da = np.einsum('bhtd,bhsd->bhts', dav, v)
        dv = np.einsum('bhts,bhtd->bhsd', a, dav)
        de = a * (da - (a * da).sum(-1, keepdims=True))
        dq = np.einsum('bhts,bhsd->bhtd', de, k) / np.sqrt(dh)
        dk = np.einsum('bhts,bhtd->bhsd', de, q) / np.sqrt(dh)
        if cfg['pe'] == 'rope':
            dq = _rope_apply(dq, ang, inverse=True)
            dk = _rope_apply(dk, ang, inverse=True)
        def merge(z):
            return z.transpose(0, 2, 1, 3).reshape(B, T, d)
        dq, dk, dv = merge(dq), merge(dk), merge(dv)
        h1 = c['h1']
        g[s + 'wq'] += np.einsum('btd,bte->de', h1, dq)
        g[s + 'wk'] += np.einsum('btd,bte->de', h1, dk)
        g[s + 'wv'] += np.einsum('btd,bte->de', h1, dv)
        dh1 = dq @ p[s + 'wq'].T + dk @ p[s + 'wk'].T + dv @ p[s + 'wv'].T
        dxi, dg1, db1 = _ln_b(dh1, c['ln1'], p[s + 'ln1g'])
        g[s + 'ln1g'] += dg1; g[s + 'ln1b'] += db1
        dx = dx + dxi
    if cfg['pe'] == 'learned':
        g['pos'][:T] += dx.sum(0)
    np.add.at(g['emb'], cache['idx'], dx)
    return g

def adam_init(p):
    return {k: np.zeros_like(v) for k, v in p.items()}, {k: np.zeros_like(v) for k, v in p.items()}

def adam_step(p, g, m, v, t, lr=3e-3):
    for k in p:
        m[k] = 0.9 * m[k] + 0.1 * g[k]
        v[k] = 0.999 * v[k] + 0.001 * g[k] ** 2
        p[k] -= lr * (m[k] / (1 - 0.9 ** t)) / (np.sqrt(v[k] / (1 - 0.999 ** t)) + 1e-8)

def train_lm(p, cfg, sample_batch, steps, lr=3e-3, log_every=0, rope_scale=1.0):
    m, v = adam_init(p)
    hist = []
    for t in range(1, steps + 1):
        idx, tgt = sample_batch()
        out = forward(p, cfg, idx, targets=tgt, rope_scale=rope_scale)
        g = backward(p, cfg, out['cache'], rope_scale=rope_scale)
        adam_step(p, g, m, v, t, lr)
        hist.append(out['loss'])
        if log_every and t % log_every == 0:
            print(f"  step {t}: loss {np.mean(hist[-log_every:]):.3f}")
    return hist

In [ ]:
V = 20
T_SEQ = 32
rn0 = np.random.default_rng(SEED % 100000)
TRANS_NEXT = np.array([rn0.permutation(V)[:4] for _ in range(V)])
TRANS_P = np.array([0.5, 0.25, 0.15, 0.1])
H_COND = -(TRANS_P * np.log(TRANS_P)).sum()

def markov_batch(B, rn, shuffle_labels=False):
    idx = np.zeros((B, T_SEQ), int)
    idx[:, 0] = rn.integers(0, V, B)
    for t in range(1, T_SEQ):
        choice = rn.choice(4, size=B, p=TRANS_P)
        idx[:, t] = TRANS_NEXT[idx[:, t - 1], choice]
    tgt = np.full((B, T_SEQ), -1)
    tgt[:, :-1] = idx[:, 1:]
    if shuffle_labels:
        for b in range(B):                       # 열 안에서 정답만 뒤섞는다
            valid = tgt[b, :-1].copy()
            tgt[b, :-1] = valid[rn.permutation(len(valid))]
    return idx, tgt

STEPS = 150 if FAST else 300
runs = {}
for causal in [True, False]:
    for shuf in [False, True]:
        p, cfg = make_config(V=V, d=48, L=2, h=4, T_max=T_SEQ, pe='learned',
                             causal=causal, seed=11)
        rn = np.random.default_rng(500)
        hist = train_lm(p, cfg, lambda: markov_batch(32, rn, shuf), steps=STEPS, lr=3e-3)
        runs[(causal, shuf)] = {'p': p, 'cfg': cfg, 'hist': np.array(hist)}
        nm = f"마스크{'O' if causal else 'X'}·레이블{'셔플' if shuf else '정상'}"
        print(f"{nm}: 최종 학습 손실 {np.mean(hist[-20:]):.3f}   ({time.time()-_t0:.0f}초)")
print(f"전이 엔트로피(정직한 하한) = {H_COND:.3f}")

---
## 2. 생성 모드의 심판 — 미래를 빼앗는다

생성 시에는 미래가 존재하지 않는다. 두 정상-레이블 모델을 **인과 마스크를 강제한
평가**(= 생성 시점의 정보 조건)로 다시 채점한다.

In [ ]:
idx_ev, tgt_ev = markov_batch(256, np.random.default_rng(77))
report = {}
for causal in [True, False]:
    r = runs[(causal, False)]
    tf_loss = forward(r['p'], r['cfg'], idx_ev, targets=tgt_ev)['loss']
    cfg_gen = dict(r['cfg']); cfg_gen['causal'] = True
    gen_loss = forward(r['p'], cfg_gen, idx_ev, targets=tgt_ev)['loss']
    report[causal] = (tf_loss, gen_loss)
    print(f"마스크{'O' if causal else 'X'} 모델:  학습 조건 손실 {tf_loss:.3f}  |  생성 조건 손실 {gen_loss:.3f}")

---
## 3. 부검 — 어텐션은 어디를 보고 있었나

In [ ]:
att = {}
for causal in [True, False]:
    r = runs[(causal, False)]
    out = forward(r['p'], r['cfg'], idx_ev[:16], want_attn=True)
    att[causal] = np.stack([a.mean(axis=(0, 1)) for a in out['attn']]).mean(0)   # (T,T)
sup = att[False][np.arange(T_SEQ - 1), np.arange(1, T_SEQ)].mean()
print(f"마스크X 모델의 평균 어텐션에서 '한 칸 미래'(대각선 위) 질량 = {sup:.2f}")

---
## 4. 교재 그림 — fig_13_5_4

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10.8, 6.9))
axes = axes.ravel()

# (a) 학습 손실 — 마스크 유무
ax = axes[0]
for causal, col, lb in [(True, CB[5], lab('마스크 O (정상)', 'masked')),
                        (False, CB[4], lab('마스크 X (누출)', 'unmasked'))]:
    h = runs[(causal, False)]['hist']
    ax.plot(np.arange(1, len(h) + 1), h, '-', color=col, lw=1.2, label=lb)
ax.axhline(H_COND, color='k', lw=0.8, ls=':')
ax.text(len(h) * 0.55, H_COND + 0.1, lab('전이 엔트로피 (정직한 하한)', 'entropy floor'), fontsize=8)
ax.set_xlabel(lab('학습 걸음', 'step'))
ax.set_ylabel(lab('학습 손실', 'train loss'))
ax.set_title(lab('(a) 버그 쪽 손실이 "더 좋다" — 하한 아래로', '(a) train loss'), fontsize=10)
ax.legend(fontsize=8)

# (b) 학습 조건 대 생성 조건
ax = axes[1]
xs = np.arange(2); w = 0.35
ax.bar(xs - w/2, [report[True][0], report[False][0]], w, color=CB[2],
       label=lab('학습 조건 (마스크는 학습 때 그대로)', 'as trained'))
ax.bar(xs + w/2, [report[True][1], report[False][1]], w, color=CB[4],
       label=lab('생성 조건 (인과 마스크 강제)', 'causal eval'))
ax.axhline(H_COND, color='k', lw=0.8, ls=':')
ax.set_xticks(xs)
ax.set_xticklabels([lab('마스크 O', 'masked'), lab('마스크 X', 'unmasked')])
ax.set_ylabel(lab('평가 손실', 'eval loss'))
ax.set_title(lab('(b) 미래를 빼앗기는 순간', '(b) generation-time loss'), fontsize=10)
ax.legend(fontsize=8)

# (c) 어텐션 부검
ax = axes[2]
imv = ax.imshow(att[False], cmap='viridis', vmin=0)
plt.colorbar(imv, ax=ax, fraction=0.046)
ax.plot([0, T_SEQ - 2], [0, T_SEQ - 2], color='w', lw=0.6, ls=':')
ax.grid(False)
ax.set_xlabel(lab('키 위치 $s$', 'key')); ax.set_ylabel(lab('질의 위치 $t$', 'query'))
ax.set_title(lab('(c) 마스크X 모델 — 질량이 대각선 위(미래)에 있다', '(c) attention autopsy'), fontsize=10)

# (d) 레이블 셔플 위생 검사
ax = axes[3]
styles = {(True, False): (CB[5], '-', lab('마스크O·정상', 'M·true')),
          (False, False): (CB[4], '-', lab('마스크X·정상', 'U·true')),
          (True, True): (CB[5], ':', lab('마스크O·셔플', 'M·shuf')),
          (False, True): (CB[4], ':', lab('마스크X·셔플', 'U·shuf'))}
for k, (col, ls, lb) in styles.items():
    h = runs[k]['hist']
    ax.plot(np.arange(1, len(h) + 1), h, ls, color=col, lw=1.3, label=lb)
ax.set_xlabel(lab('학습 걸음', 'step'))
ax.set_ylabel(lab('학습 손실', 'train loss'))
ax.set_title(lab('(d) 셔플하면 베낄 미래도 사라진다', '(d) shuffle hygiene check'), fontsize=10)
ax.legend(fontsize=8)

save_book_fig(fig, 'fig_13_5_4')
plt.show()

> ### 읽는 법
>
> (a) 마스크를 빠뜨린 모델의 학습 손실은 정상 모델보다 **낮다** — 심지어 데이터의
> 전이 엔트로피(정직한 하한) 아래로 내려간다. 하한 아래의 손실은 그 자체로 누출의
> 지문이다.
> (b) 생성 조건(미래 없음)으로 채점하는 순간 처지가 뒤집힌다. 정상 모델은 두 조건의
> 손실이 같고(애초에 미래를 안 썼으므로), 누출 모델은 폭발한다.
> (c) 부검 결과: 누출 모델의 어텐션 질량은 대각선 바로 위 — 정답이 입력으로 보이는
> 자리 — 에 몰려 있다.
> (d) 위생 검사: 정답 열을 열 안에서 섞으면 베낄 미래도 함께 사라지므로, 누출 모델의
> 우위가 소멸한다. (셔플 후에도 마스크X가 아주 약간 낮게 머무는 것까지 설명된다 —
> 셔플된 정답은 같은 열의 토큰이므로, 열 전체를 보는 쪽이 그 주변 분포를 더 잘
> 추정한다. 누출의 잔향이다.) **"손실이 너무 좋다"는 의심스러운 신호이고, 셔플
> 검사는 그 의심을 몇 분에 판정한다**(§13.5.5).

---
## 5. 자기 점검

1. (a)에서 누출 모델의 손실이 0이 아니라 어떤 값에서 멈추는 이유는 무엇인가? (힌트: 마지막 위치의 정답은 어디에도 없다.)
2. §13.5.5의 세 번째 진단(가시성 검사)을 구현하라: $\partial y_t/\partial x_s$ ($s>t$)가 마스크O 모델에서 정확히 0인지 확인하는 코드를 짜 보라.
3. 마스크를 "한 칸 부족하게"(대각선 포함을 잘못 처리해 자기 자신을 못 보게) 만들면 어떤 증상이 나는가? 반대로 대각선 위 한 칸만 뚫리면?
4. (c)의 정상 모델 버전을 그려 비교하라. 마르코프 연쇄라는 데이터 특성상 질량이 어디에 몰려야 정상인가?

## 6. 직접 바꿔 볼 손잡이

| 손잡이 | 위치 | 기본값 | 바꾸면 |
|---|---|---|---|
| `TRANS_P` | 1절 | [.5,.25,.15,.1] | 전이 엔트로피(하한)의 위치 |
| `V`, `T_SEQ` | 1절 | 20, 32 | 과제 규모 |
| `STEPS` | 1절 | 300 | 누출 학습의 속도 |

In [ ]:
print(f"총 실행 시간: {time.time() - _t0:.1f}초")